In [1]:
# -- Use the following line for google colab removing the hash at the beginning.
! pip install -q 'corner==2.2.2' 'bilby==2.2.2' 'astropy==6.0.1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.9/102.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 1.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.

## Initialization

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd
import corner

In [3]:
!pip install healpy astropy numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 31.1 MB/s eta 0:00:00


In [5]:
# pip install healpy astropy numpy
import json, numpy as np, healpy as hp
from astropy.io import fits

URL = "https://dcc.ligo.org/LIGO-P2000230/public/GW190814_skymap.fits.gz"
CREDIBLE = 0.90

def hpd_threshold(prob, level):
    flat = prob.ravel()
    order = np.argsort(flat)[::-1]
    csum = np.cumsum(flat[order])
    t = flat[order[np.searchsorted(csum, level * flat.sum())]]
    return t

def healpix_components(mask, nside, nest=True):
    npix = mask.size
    visited = np.zeros(npix, bool)
    comps = []
    idxs = np.where(mask)[0]
    idxset = set(idxs.tolist())
    for s in idxs:
        if visited[s]: continue
        q=[int(s)]; visited[s]=True; comp=[int(s)]
        while q:
            p=q.pop()
            neigh = hp.get_all_neighbours(nside, p, nest=nest)
            for nb in neigh[neigh>=0]:
                if (nb in idxset) and not visited[nb]:
                    visited[nb]=True; q.append(int(nb)); comp.append(int(nb))
        comps.append(comp)
    return comps

def minimal_ra_span(ra_deg):
    ra = np.mod(ra_deg, 360.0)
    s = np.sort(ra); dbl = np.concatenate([s, s+360])
    N=len(s); best=(1e9,0,0)
    for i in range(N):
        j=i+N-1
        w=dbl[j]-dbl[i]
        if w<best[0]:
            a=(dbl[i])%360.0; b=(a+w)%360.0; best=(w,a,b)
    return best[1], best[2]  # ra_min, ra_max (eastward shortest interval)

with fits.open(URL, memmap=True) as h:
    tab = h[1].data; hdr = h[1].header
    prob  = np.asarray(tab["PROB"], float)
    has3d = all(k in tab.columns.names for k in ["DISTMU","DISTSIGMA","DISTNORM"])
    if has3d:
        mu = np.asarray(tab["DISTMU"], float)
        sg = np.asarray(tab["DISTSIGMA"], float)
        nn = np.asarray(tab["DISTNORM"], float)
    nside = hdr["NSIDE"]
    nest  = hdr.get("ORDERING","NESTED").upper().startswith("NEST")

# --- 90% HPD mask and main island ---
thr   = hpd_threshold(prob, CREDIBLE)
mask  = prob >= thr
comps = healpix_components(mask, nside, nest=nest)
main  = np.array(comps[np.argmax([prob[c].sum() for c in comps])], int)

# --- RA/Dec bounds (ICRS) ---
theta, phi = hp.pix2ang(nside, main, nest=nest)  # theta colat, phi RA
dec = 90 - np.degrees(theta)
ra  = np.degrees(phi) % 360.0
dec_min, dec_max = float(dec.min()), float(dec.max())
ra_min,  ra_max  = minimal_ra_span(ra)

# --- Distance 90% (island-marginal) ---
dist_p5 = dist_p95 = None
if has3d:
    r = np.linspace(0, max(np.max(mu[main]+5*sg[main]), 50.0), 4000)
    r2 = r*r
    w  = prob[main]
    pr = (w[:,None] * nn[main,None] * r2[None,:] *
          np.exp(-0.5*((r[None,:]-mu[main,None])/np.clip(sg[main,None],1e-6,None))**2)).sum(0)
    pr /= np.trapz(pr, r)
    cdf = np.cumsum(pr)*(r[1]-r[0])
    dist_p5, dist_p95 = float(np.interp(0.05, cdf, r)), float(np.interp(0.95, cdf, r))

out = {
  "event": "GW190814",
  "credible_level": CREDIBLE,
  "main_island_probability_mass": float(prob[main].sum()),
  "pixels_in_island": int(main.size),
  "ra_min_deg_eastward_interval": ra_min,
  "ra_max_deg_eastward_interval": ra_max,
  "dec_min_deg": dec_min,
  "dec_max_deg": dec_max
}
if dist_p5 is not None:
    out["distance_5th_percentile_Mpc"]  = dist_p5
    out["distance_95th_percentile_Mpc"] = dist_p95

print(json.dumps(out, indent=2))
with open("GW190814_main_island_90pct_bounds.json","w") as f: json.dump(out, f, indent=2)
print("Saved GW190814_main_island_90pct_bounds.json")


{
  "event": "GW190814",
  "credible_level": 0.9,
  "main_island_probability_mass": 0.8246973886079083,
  "pixels_in_island": 4212,
  "ra_min_deg_eastward_interval": 10.239257812499998,
  "ra_max_deg_eastward_interval": 14.897460937499998,
  "dec_min_deg": -27.321590475560456,
  "dec_max_deg": -22.225648048388607,
  "distance_5th_percentile_Mpc": 196.53538512435878,
  "distance_95th_percentile_Mpc": 284.57127373172665
}
Saved GW190814_main_island_90pct_bounds.json


In [7]:
!pip install pyvo astropy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.5 MB/s eta 0:00:00


In [8]:
# pip install pyvo astropy
import numpy as np, pyvo as vo
from astropy.cosmology import Planck18 as cosmo

# ---- your wedge (use the numbers you printed) ----
ra_min, ra_max = 10.2392578125, 14.8974609375      # deg (eastward-shortest interval)
dec_min, dec_max = -27.3215904756, -22.2256480484  # deg
dl_min_mpc, dl_max_mpc = 196.5353851, 284.5712737  # Mpc

# 1) distance -> z window (coarse; good enough for preselection)
#    We invert luminosity distance with a 1D search.
from astropy import units as u
def z_at_dl(DL_mpc):
    # robust 1D search in [0,0.3] (more than enough for ~300 Mpc)
    zgrid = np.linspace(0.0, 0.3, 2001)
    dl = cosmo.luminosity_distance(zgrid).to(u.Mpc).value
    return float(np.interp(DL_mpc, dl, zgrid))
zmin = z_at_dl(dl_min_mpc)
zmax = z_at_dl(dl_max_mpc)
print({"z_window_est": [zmin, zmax]})  # ~0.047–0.067 for your numbers

# 2) Build ADQL where-clause pieces
def ra_box_clause(ra_min, ra_max):
    # handle wrap: if min<=max: single clause; else split into two
    if ra_min <= ra_max:
        return f"(ra BETWEEN {ra_min} AND {ra_max})"
    else:
        return f"((ra >= {ra_min}) OR (ra <= {ra_max}))"

where_sky = f"""{ra_box_clause(ra_min, ra_max)} AND
(dec BETWEEN {dec_min} AND {dec_max})"""

# --- Choose a DES DR2 table and star/galaxy criterion ---
# Typical DR2 wide table is often 'des_dr2.main' with SExtractor columns
# like spread_model_{band}. If your account/browser shows a different
# table name or columns in Data Lab, adjust the names below accordingly.
table = "des_dr2.main"

# Preferred classifier if present (e.g. in GOLD tables):
#    EXTENDED_CLASS_MASH_SOF in {0,1,2} => galaxy-like (>=1 usually extended)
# Fallback using SPREAD_MODEL in i-band (classic DES cut; tweak as you like):
sg_cut = "(spread_model_i + 3*spreaderr_model_i > 0.005)"  # galaxy-like

# Optional photo-z column varies by table. If available (e.g., des_dr2.photoz)
# join it; otherwise skip the z cut (or do it later by crossmatching).
use_photoz = False    # set True if you know a photo-z column to use in this table
photoz_col = "photo_z_mean"

if use_photoz:
    where_z = f"({photoz_col} BETWEEN {zmin} AND {zmax})"
else:
    where_z = "1=1"   # no photo-z constraint here

# 3) Compose ADQL
adql = f"""
SELECT
  COUNT(*) AS n_all,
  SUM(CASE WHEN {sg_cut} THEN 1 ELSE 0 END) AS n_gal_like
FROM {table}
WHERE
  {where_sky}
  AND {where_z}
"""

print("ADQL:\n", adql)

# 4) Run via TAP
svc = vo.dal.TAPService("https://datalab.noirlab.edu/tap")
res = svc.search(adql)
row = list(res)[0]
print({"n_all": int(row["n_all"]), "n_gal_like": int(row["n_gal_like"])})

# 5) If you want the actual rows (e.g., first 50 galaxy-like objects):
adql_list = f"""
SELECT TOP 50
  coadd_object_id, ra, dec,
  mag_auto_i, spread_model_i, spreaderr_model_i
FROM {table}
WHERE
  {where_sky}
  AND {sg_cut}
  AND {where_z}
ORDER BY mag_auto_i ASC
"""
res2 = svc.search(adql_list)
print("Example rows:", len(res2))

{'z_window_est': [0.04296307718019475, 0.061399269574667355]}
ADQL:
 
SELECT
  COUNT(*) AS n_all,
  SUM(CASE WHEN (spread_model_i + 3*spreaderr_model_i > 0.005) THEN 1 ELSE 0 END) AS n_gal_like
FROM des_dr2.main
WHERE
  (ra BETWEEN 10.2392578125 AND 14.8974609375) AND
(dec BETWEEN -27.3215904756 AND -22.2256480484)
  AND 1=1

{'n_all': 3033756, 'n_gal_like': 2}
Example rows: 50


In [12]:
adql = """
SELECT TOP 20
  coadd_object_id, ra, dec,
  mag_auto_i, flags_i,
  wavg_spread_model_i, wavg_spreaderr_model_i,
  spread_model_i,      spreaderr_model_i
FROM des_dr2.main
WHERE
  (ra BETWEEN 10.2392578125 AND 14.8974609375)
  AND (dec BETWEEN -27.3215904756 AND -22.2256480484)
ORDER BY ra
"""
rows = svc.search(adql)
print("Rows:", len(rows))
for r in rows:
    print(dict(r))



Rows: 20
{'coadd_object_id': 1091122976, 'ra': 10.239258, 'dec': -22.544756, 'mag_auto_i': 24.511843, 'flags_i': 0, 'wavg_spread_model_i': -99.0, 'wavg_spreaderr_model_i': -99.0, 'spread_model_i': 0.011796553, 'spreaderr_model_i': 0.011987626}
{'coadd_object_id': 1092097065, 'ra': 10.239259, 'dec': -24.541187, 'mag_auto_i': 23.889456, 'flags_i': 0, 'wavg_spread_model_i': -0.0015679243, 'wavg_spreaderr_model_i': 0.004354562, 'spread_model_i': -0.0019822777, 'spreaderr_model_i': 0.00437754}
{'coadd_object_id': 1087689173, 'ra': 10.23926, 'dec': -26.411695, 'mag_auto_i': 21.286604, 'flags_i': 0, 'wavg_spread_model_i': 0.010400429, 'wavg_spreaderr_model_i': 0.00060704886, 'spread_model_i': 0.009670754, 'spreaderr_model_i': 0.00052798894}
{'coadd_object_id': 1088673742, 'ra': 10.23926, 'dec': -23.438308, 'mag_auto_i': 22.500072, 'flags_i': 1, 'wavg_spread_model_i': 0.00950184, 'wavg_spreaderr_model_i': 0.0023776689, 'spread_model_i': 0.01387239, 'spreaderr_model_i': 0.0023408623}
{'coadd_ob

In [17]:
# pip install healpy astropy numpy
import numpy as np, healpy as hp
from astropy.io import fits

URL = "https://dcc.ligo.org/LIGO-P2000230/public/GW190814_skymap.fits.gz"
CREDIBLE = 0.90

def hpd_threshold(prob, level):
    flat = prob.ravel()
    order = np.argsort(flat)[::-1]
    csum = np.cumsum(flat[order])
    t = flat[order[np.searchsorted(csum, level * flat.sum())]]
    return t

def healpix_components(mask, nside, nest=True):
    npix = mask.size
    visited = np.zeros(npix, bool)
    comps = []
    idxs = np.where(mask)[0]
    idxset = set(idxs.tolist())
    for s in idxs:
        if visited[s]: continue
        q=[int(s)]; visited[s]=True; comp=[int(s)]
        while q:
            p=q.pop()
            neigh = hp.get_all_neighbours(nside, p, nest=nest)
            for nb in neigh[neigh>=0]:
                if (nb in idxset) and not visited[nb]:
                    visited[nb]=True; q.append(int(nb)); comp.append(int(nb))
        comps.append(comp)
    return comps

with fits.open(URL, memmap=True) as h:
    tab = h[1].data; hdr = h[1].header
    prob  = np.asarray(tab["PROB"], float)
    nside = int(hdr["NSIDE"])
    nest  = hdr.get("ORDERING","NESTED").upper().startswith("NEST")

thr   = hpd_threshold(prob, CREDIBLE)
mask  = prob >= thr
comps = healpix_components(mask, nside, nest=nest)
main  = np.array(comps[np.argmax([prob[c].sum() for c in comps])], dtype=int)

fname = f"GW190814_main_island_nside{nside}_nest{int(nest)}.npy"
np.save(fname, main)
print("Saved:", fname, "  pixels:", main.size, "  prob_mass:", prob[main].sum())


Saved: GW190814_main_island_nside1024_nest1.npy   pixels: 4212   prob_mass: 0.8246973886079083


In [18]:
import numpy as np, healpy as hp

# Use the exact name printed above
island_pix = np.load("GW190814_main_island_nside1024_nest1.npy")  # <-- adjust if your nside/nest differ
island_set = set(island_pix.tolist())

theta = np.radians(90.0 - np.array(tbl["dec"]))
phi   = np.radians(np.array(tbl["ra"]) % 360.0)
pix   = hp.ang2pix(1024, theta, phi, nest=True)  # <-- make sure NSIDE and nest match the file/header

in_island = np.fromiter((p in island_set for p in pix), count=len(pix), dtype=bool)
tbl_island = tbl[in_island]
print("Inside island:", len(tbl_island))


Inside island: 1393743


In [19]:
wavg_valid = (tbl_island["wavg_spread_model_i"] > -98) & (tbl_island["wavg_spreaderr_model_i"] > -98)
morph_3sig = np.where(
    wavg_valid,
    tbl_island["wavg_spread_model_i"] + 3*tbl_island["wavg_spreaderr_model_i"],
    tbl_island["spread_model_i"]      + 3*tbl_island["spreaderr_model_i"]
)
gal_mask = morph_3sig > 0.005
gal_tbl  = tbl_island[gal_mask]
print({"inside_island": len(tbl_island), "island_galaxies": len(gal_tbl)})

{'inside_island': 1393743, 'island_galaxies': 1307209}


In [20]:
# make a 50% HPD mask + main island, analogous to your 90% code
CREDIBLE = 0.50
thr50   = hpd_threshold(prob, CREDIBLE)
mask50  = prob >= thr50
comps50 = healpix_components(mask50, nside, nest=True)
main50  = np.array(comps50[np.argmax([prob[c].sum() for c in comps50])], int)
island50_set = set(main50.tolist())

pix50 = hp.ang2pix(nside,
                   np.radians(90.0 - np.array(tbl["dec"])),
                   np.radians(np.array(tbl["ra"]) % 360.0),
                   nest=nest)
in_island50 = np.fromiter((p in island50_set for p in pix50), count=len(pix50), dtype=bool)
tbl_50 = tbl[in_island50]
print({"inside_50pct_island": len(tbl_50)})


{'inside_50pct_island': 398602}


In [21]:
t = tbl_island  # or tbl_50 if you did A
q = (t["flags_i"] == 0) & (t["mag_auto_i"] < 23.0)  # stricter mag + FLAGS
tq = t[q]

wavg_ok = (tq["wavg_spread_model_i"] > -98) & (tq["wavg_spreaderr_model_i"] > -98)
m3 = np.where(
    wavg_ok,
    tq["wavg_spread_model_i"] + 3*tq["wavg_spreaderr_model_i"],
    tq["spread_model_i"]      + 3*tq["spreaderr_model_i"]
)
gal = tq[m3 > 0.010]  # slightly stricter than 0.005
print({"after_quality": len(tq), "galaxies_strict": len(gal)})


{'after_quality': 337190, 'galaxies_strict': 246921}


In [22]:
SELECT g.coadd_object_id, g.ra, g.dec, p.photoz_mean
FROM des_dr2.main AS g
JOIN des_dr2.photoz AS p USING (coadd_object_id)
WHERE ...sky cuts...
  AND p.photoz_mean BETWEEN 0.043 AND 0.061
  AND g.flags_i = 0
  AND g.mag_auto_i < 23.0
  AND (morphology cut…)


SyntaxError: invalid character '…' (U+2026) (ipython-input-2883140081.py, line 8)

In [27]:
# Assumes you already have:
#   - svc  -> pyvo.dal.TAPService("https://datalab.noirlab.edu/tap")
#   - the island .npy saved (file name below)
#   - healpy, numpy installed

import numpy as np, healpy as hp

# --- 1) ADQL join: DES DR2 main ⨝ y6_gold with photo-z (dnf_z) + quality in the rectangle ---
adql = """
SELECT
  g.coadd_object_id AS coadd_object_id,
  g.ra, g.dec,
  g.mag_auto_i, g.flags_i,
  y.dnf_z,
  y.wavg_spread_model_z, y.wavg_spreaderr_model_z,
  y.spread_model_z,      y.spreaderr_model_z
FROM des_dr2.main AS g
JOIN des_dr2.y6_gold AS y
  ON g.coadd_object_id = y.coadd_object_id
WHERE
  (g.ra  BETWEEN 10.2392578125 AND 14.8974609375)
  AND (g.dec BETWEEN -27.3215904756 AND -22.2256480484)
  AND (g.flags_i < 4)
  AND (g.mag_auto_i BETWEEN 16 AND 24.5)
  AND (y.dnf_z BETWEEN 0.043 AND 0.061)
"""
t = svc.search(adql).to_table()
print("Rows in rectangle (photo-z + quality):", len(t))

# --- 2) Exact 90% main island mask (set filename + NSIDE/NEST to match your skymap) ---
island_fname = "GW190814_main_island_nside1024_nest1.npy"  # change if your file name differs
nside = 1024
nest  = True

island_pix = np.load(island_fname)
theta = np.radians(90.0 - np.array(t["dec"]))
phi   = np.radians(np.array(t["ra"]) % 360.0)
pix   = hp.ang2pix(nside, theta, phi, nest=nest)
mask_island = np.isin(pix, island_pix)
ti = t[mask_island]
print("Inside 90% main island:", len(ti))

# --- 3) Star/galaxy morphology cut (z-band; use weighted if valid else fallback) ---
wok = (ti["wavg_spread_model_z"] > -98) & (ti["wavg_spreaderr_model_z"] > -98)
m3  = np.where(
    wok,
    ti["wavg_spread_model_z"] + 3*ti["wavg_spreaderr_model_z"],
    ti["spread_model_z"]      + 3*ti["spreaderr_model_z"]
)
gal = ti[m3 > 0.005]  # tighten to 0.010 if you want fewer, purer galaxies
print({"in_island_photoz": len(ti), "galaxy_like": len(gal)})

# --- 4) Save CSV for quick inspection ---
try:
    import pandas as pd
    df = gal.to_pandas()
    df.to_csv("GW190814_y6gold_photoz_galaxies.csv", index=False)
    print("Saved GW190814_y6gold_photoz_galaxies.csv with", len(df), "rows")
except Exception as e:
    print("CSV save skipped (pandas not installed?):", e)



Rows in rectangle (photo-z + quality): 1934
Inside 90% main island: 1341
{'in_island_photoz': 1341, 'galaxy_like': 589}
Saved GW190814_y6gold_photoz_galaxies.csv with 589 rows


In [28]:
# Ranking the 589 by sky prob × distance likelihood × brightness
import numpy as np, pandas as pd, healpy as hp
from astropy.cosmology import Planck18 as cosmo
from astropy import units as u
from astropy.io import fits

# 1) Load the GW190814 skymap to get prob/dist fields
URL = "https://dcc.ligo.org/LIGO-P2000230/public/GW190814_skymap.fits.gz"
with fits.open(URL, memmap=True) as h:
    tab = h[1].data
    hdr = h[1].header
    nside = int(hdr["NSIDE"])
    nest  = hdr.get("ORDERING","NESTED").upper().startswith("NEST")
    prob      = np.asarray(tab["PROB"], float)
    distmu    = np.asarray(tab["DISTMU"], float)
    distsigma = np.asarray(tab["DISTSIGMA"], float)

# 2) Load your 589-table (from the previous step)
gal = pd.read_csv("GW190814_y6gold_photoz_galaxies.csv")

# 3) Per-galaxy HEALPix pixel
theta = np.radians(90.0 - gal["dec"].values)
phi   = np.radians(gal["ra"].values % 360.0)
pix   = hp.ang2pix(nside, theta, phi, nest=nest)

# 4) Sky weight
p_sky = prob[pix]

# 5) Distance weight from dnf_z
r_gal = cosmo.luminosity_distance(gal["dnf_z"].values).to(u.Mpc).value
mu    = distmu[pix]
sg    = np.clip(distsigma[pix], 1e-6, None)
p_r   = np.exp(-0.5 * ((r_gal - mu)/sg)**2)   # Gaussian-in-r proxy for ranking

# 6) Brightness weight (simple)
mag = gal["mag_auto_i"].values
w_mag = 10**(-0.4*(mag - 20.0))               # pivot at i~20

# 7) Combined score and export Top-200
score = p_sky * p_r * w_mag
ranked = gal.copy()
ranked["p_sky"] = p_sky
ranked["p_r"]   = p_r
ranked["score"] = score
ranked = ranked.sort_values("score", ascending=False)

topN = 200 if len(ranked) >= 200 else len(ranked)
ranked.head(topN).to_csv("GW190814_DES_top200_ranked.csv", index=False)
print(f"Saved Top-{topN}: GW190814_DES_top200_ranked.csv")


Saved Top-200: GW190814_DES_top200_ranked.csv
